In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("Titanic-Dataset.csv") if os.path.exists("Titanic-Dataset.csv") else pd.read_csv("https://www.kaggle.com/datasets/yasserh/titanic-dataset")

df = df.drop(columns=['Cabin'], errors='ignore')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop_duplicates()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)
df['Pclass_encoded'] = LabelEncoder().fit_transform(df['Pclass'].astype(str))

num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
df[[c + '_scaled' for c in num_cols]] = StandardScaler().fit_transform(df[num_cols])

Q1, Q3 = df['Fare'].quantile([0.25, 0.75])
df['Fare'] = df['Fare'].clip(lower=Q1 - 1.5*(Q3-Q1), upper=Q3 + 1.5*(Q3-Q1))

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Title'] = df['Name'].str.extract(r' ([a-z]+)\.', expand=False).fillna('unknown')
df = pd.get_dummies(df, columns=['Title'], drop_first=True, dtype=int)

drops = [c for c in ['Survived', 'Name', 'Ticket', 'Pclass', 'PassengerId'] if c in df.columns]
X = df.drop(columns=drops)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
import joblib

df = pd.read_csv("Titanic-Dataset.csv")
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

joblib.dump(gb, 'gb_model.pkl')

['gb_model.pkl']

In [4]:
import pandas as pd
import joblib

model = joblib.load('gb_model.pkl')

test = pd.read_csv('Titanic-Dataset.csv')
test['Age'] = test['Age'].fillna(test['Age'].median())
test['Fare'] = test['Fare'].fillna(test['Fare'].median())
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

X = test[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]
test['Survived'] = model.predict(X)

test[['PassengerId', 'Survived']].to_csv('submission.csv', index=False)

In [6]:
import joblib
model = joblib.load('gb_model.pkl')

In [ ]:
!git init
!git remote add origin git@github.com:lavignenancy/titanic-gradient-boosting.git
!git branch -M main
!git add gradient_boosting.ipynb
!git commit -m "Initial commit Titanic ML project"
!git push -u origin main
